In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [94]:
electionresults1 = pd.read_csv('Queries/electionresults2.csv')
electionresults1.columns


Index(['candidate_id', 'candidate', 'date', 'race_id', 'race_name',
       'candidate_party', 'votes', 'is_winner', 'election_type',
       'is_uncontested', 'number_of_seats_in_election',
       'total_number_of_ballots_in_race'],
      dtype='str')

In [82]:
candidatedata = pd.read_csv('Queries/Candidate_Data_Query2.csv')
candidatedata.columns


Index(['hubspot_id', 'state', 'office_level', 'office_type',
       'candidate_office', 'election_date', 'is_uncontested',
       'viability_score', 'general_election_result', 'general_votes_received',
       'total_general_votes_cast'],
      dtype='str')

In [95]:

def classify_office(race_name: str) -> str:
    x = (race_name or "").lower()

    # ---------- Federal ----------
    if re.search(r"\bu\.s\.\s*(house|senate)\b|\bcongress\b|\bpresident\b|\bfederal\b", x):
        return "Federal"

    # ---------- State ----------
    if re.search(r"\bgovernor\b|\battorney general\b|\bsecretary of state\b|\bstate (senate|house|representative|rep)\b|\bstate board\b", x):
        return "state"

    # ---------- County (general government) ----------
    # Avoid treating special districts that contain "county" (e.g., "Yakima County Fire ...") as county government.
    if re.search(r"\bcounty (commission|commissioner|council|board|board of supervisors|supervisor|executive)\b", x) and not re.search(
        r"\b(fire|sewer|water|port|park|library|road|conservation|soil)\b.*\bdistrict\b", x
    ):
        return "county"
    if re.search(r"\bsheriff\b|\bprosecut(or|ing)\b|\bdistrict attorney\b|\bcounty clerk\b|\bcounty treasurer\b|\bcounty assessor\b|\bregister of deeds\b", x):
        return "county"

    # ---------- Township / Town (WI-style town government) ----------
    if re.search(r"\btownship\b|\bcharter township\b", x):
        return "township"
    if re.search(r"\btown board\b|\btown clerk\b|\btown treasurer\b|\btown supervisor\b|\btown chair(person)?\b", x):
        return "township"
    if re.search(r"\btownship trustee\b|\btownship trustee board\b", x):
        return "township"

    # ---------- Local (education districts) ----------
    if re.search(r"\bschool district\b|\bschool board\b|\bboard of education\b|\bcommunity (unit|consolidated) school district\b", x):
        return "local"
    if re.search(r"\bcommunity college district\b|\bjunior college district\b|\bcollege district\b|\bregents\b", x):
        return "local"

    # ---------- City / municipal ----------
    # Covers: councilmember/councilor, city commission, borough/village/town council,
    # CT selectman + representative town meeting, municipal assembly, etc.
    if re.search(
        r"\bmayor\b|\bcity council\b|\bcouncil(member|or|men)?\b|\bcity commission\b|\bborough council\b|\bvillage\b|\bassembly\b|\btown council\b"
        r"|\bfirst selectman\b|\bselectman\b|\brepresentative town meeting\b|\bboard of finance\b|\bconstable\b|\btown trustee\b",
        x
    ):
        return "city"

    # ---------- Judicial (keep as other in your scheme) ----------
    if re.search(r"\bmunicipal court\b|\bcourt\b|\bjudge\b|\bjustice of the peace\b|\bmagistrate\b", x):
        return "other"

    # ---------- Special districts / quasi-government (stay as other) ----------
    # library / park / sewer / port / fire / conservation / road / health, etc.
    if re.search(r"\bdistrict\b", x) and re.search(r"\b(commissioner|board|trustee|director|supervisor|regent)\b", x):
        return "other"
    if re.search(
        r"\blibrary district\b|\bpark district\b|\bsewer district\b|\bwater district\b|\bport of\b"
        r"|\bfire protection district\b|\brural fire\b|\bsoil and water\b|\bconservation district\b|\broad dist\b|\broad district\b|\bhealth board\b",
        x
    ):
        return "other"

    return "other"

# apply
electionresults1["level"] = electionresults1["race_name"].apply(classify_office)



In [96]:
electionresults1["level"].value_counts()

level
city        486
township    189
local       178
other       132
Federal      14
Name: count, dtype: int64

In [100]:
level_counts = (
    electionresults1["level"]
    .value_counts(dropna=False)
    .rename_axis("level")
    .reset_index(name="count")
)

level_counts["percent"] = (level_counts["count"] / level_counts["count"].sum() * 100).round(1)

level_counts = level_counts.sort_values("count", ascending=False)
level_counts


,level,count,percent
0,city,486,48.6
1,township,189,18.9
2,local,178,17.8
3,other,132,13.2
4,Federal,14,1.4


In [99]:
electionresults1["race_id"].value_counts()

race_id
205583    13
367952    13
415368     8
368338     7
406367     7
          ..
414757     1
239963     1
204871     1
414502     1
405898     1
Name: count, Length: 473, dtype: int64

In [98]:
race_size_counts = (
    electionresults1["race_id"]
    .value_counts()          # races -> number of candidate rows per race (or observations per race_id)
    .value_counts()          # frequency of those race sizes
    .rename_axis("num_candidates_per_race_id")
    .reset_index(name="num_race_ids")
)

race_size_counts["percent"] = (
    race_size_counts["num_race_ids"] / race_size_counts["num_race_ids"].sum() * 100
).round(1)

race_size_counts = race_size_counts.sort_values("num_candidates_per_race_id")
race_size_counts

,num_candidates_per_race_id,num_race_ids,percent
0,1,212,44.8
1,2,133,28.1
2,3,57,12.1
3,4,38,8.0
4,5,19,4.0
5,6,8,1.7
6,7,3,0.6
8,8,1,0.2
7,13,2,0.4


In [43]:
electionresults1.loc[electionresults1["level"] == "other", "race_name"].tolist()


['OK Town of Tipton Board of Trustees Office 1 Special',
 'IL Gail Borden Public Library District Trustee Board At-large',
 'WA Midway Sewer District Commissioner Board Position 2',
 'IL Brookeridge Park District Commissioner Board At-large',
 'WA Port of Camas-Washougal District Commissioner Board 1',
 'WA Yakima County Fire  Commissioner Board Position 2 4',
 'NM Upper Chama Soil and Water Conservation District Supervisor Board At-large',
 'NM Coronado Soil and Water Conservation District Supervisor Board At-large',
 'IL Warrenville Fire Protection District Trustee Board At-large',
 'OR Wy’East Rural Fire Protection District Director',
 'MA Dartmouth Town Health Board',
 'OR Position 3 Commissioner – Bear Creek Hideout #2 Road Dist.',
 'CA San Diego County Supervisor 1',
 'IL Homer Community Library Trustee Board At-large Special',
 'WI Verona Alderperson 4',
 'IL Bloomingdale Public Library Trustee Board At-large',
 'IL Lake Villa Public Library District Trustee Board At-large',
 'F

In [44]:
electionresults1["is_winner"].value_counts()

is_winner
Y    746
N    254
Name: count, dtype: int64

In [47]:
electionresults1[["is_uncontested","is_winner"]].value_counts()

is_uncontested  is_winner
True            Y            440
False           Y            306
                N            254
Name: count, dtype: int64

In [93]:


combo_counts = (
    electionresults1[["is_uncontested", "is_winner"]]
    .value_counts(dropna=False)
    .rename("count")
    .reset_index()
)

combo_counts["percent"] = (combo_counts["count"] / combo_counts["count"].sum() * 100).round(1)

# nice ordering (optional)
combo_counts = combo_counts.sort_values(["is_uncontested", "is_winner"], ascending=[True, False])

combo_counts


,is_uncontested,is_winner,count,percent
1,False,Y,306,30.6
2,False,N,254,25.4
0,True,Y,440,44.0


In [77]:
electionresults1['number_of_seats_in_election'].value_counts().sort_index()

number_of_seats_in_election
1     457
2     182
3     188
4     139
5      25
6       2
7       2
8       2
9       1
11      1
15      1
Name: count, dtype: int64

candidate_office
school board                         7
Trustee                              5
City council                         5
School Board                         4
La Crosse School Board               3
                                    ..
Fraser City Council                  1
Harvard 50 School Board              1
Bloomingdale Village Board           1
Quincy City Council - Position 4     1
Westchester Village Library Board    1
Name: count, Length: 909, dtype: int64

In [102]:
candidatedata["hubspot_id"].value_counts()


hubspot_id
37409908778    3
35255886648    2
35424553378    2
34285382928    2
33469965334    1
              ..
35629789778    1
28851891727    1
32677961758    1
36059144010    1
33221537919    1
Name: count, Length: 995, dtype: int64

In [ ]:
candidatedata["office_"]

0      School Board
1      City Council
2      Town Council
3             Mayor
4      School Board
           ...     
995    Town Council
996    School Board
997    School Board
998    School Board
999           Other
Name: office_type, Length: 1000, dtype: str